In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import warnings
warnings.filterwarnings('ignore')
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import string
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
df = pd.read_csv("data/movies_metadata.csv")

In [3]:
df.head()

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


In [ ]:
print(df.shape)
print(df.info())
print(df.isnull().sum())

print("Duplicated:",df.duplicated().sum())
print(df.columns)

In [ ]:
df['status'].value_counts()

In [ ]:
genres = df.iloc[0]['genres']
genres

In [ ]:
# Pre processing
columns = ['id','title','genres','overview', 'tagline', 'original_language','popularity',
           'vote_average','status' ]

df = df[columns]
df = df.drop_duplicates().reset_index(drop= True)

#Genre extraction
def genre_extract(genre_ext):
  genre = ast.literal_eval(genre_ext)
  return " ".join([i['name'] for i in genre])

df = df.dropna(subset=['title'])
df['overview'] = df['overview'].fillna(' ')
df['genres'] = df['genres'].apply(genre_extract)
df['status'] = df['status'].fillna('no Update')
df['tagline'] = df['tagline'].fillna(' ')
df['original_language'] = df['original_language'].fillna(df['original_language'].mode()[0])
df.head()

In [ ]:
#Creating a new column with everything in one place
df['tags'] = df['overview'] + " " + df['tagline'] + " " + df['genres']

In [ ]:
print(df['tags'][0])
print(df['tags'][1])
print(df['tags'][2])

In [ ]:
nltk.download('stopwords')
nltk.download('wordnet')

In [ ]:
# Remove punchuations, stopwords, make it in lower case
#lower case
df['tags'] = df['tags'].apply(lambda x: x.lower())

#removing punchuation
df['tags'] = df['tags'].apply(lambda x: x.translate(str.maketrans('','',string.punctuation)))

#removing stopwords
stop_words = set(stopwords.words('english'))
df['tags'] = df['tags'].apply(lambda x: ' '.join(i for i in x.split() if not i in stop_words))

# Lematizing the sentence -> like end & ending. convert ending to its base form end
lematizer = WordNetLemmatizer()

df['tags'] = df['tags'].apply(lambda words: ' '.join(lematizer.lemmatize(word) for word in words.split()))

df = df.reset_index(drop= True)


In [ ]:
print(df['tags'][0])
print(df['tags'][1])
print(df['tags'][2])

In [ ]:
# vectorizing with tf idf
tf_idf = TfidfVectorizer(max_features= 50000, ngram_range=(1,2), stop_words= 'english')
# Although I have removed the stopwords but still using it to be safe

# Vectorizing
tf_idf_matrix = tf_idf.fit_transform(df['tags'])



In [ ]:
print(tf_idf_matrix.shape)

In [ ]:

indices = pd.Series(df.index , df['title']).drop_duplicates()
indices

In [ ]:
sim_score = cosine_similarity(tf_idf_matrix[3], tf_idf_matrix).flatten()
x = sorted(sim_score, reverse= True)[:5]
print(x)

In [ ]:
def reccomend(title, n=10):
  if title not in indices:
    return "Movie not found"
  idx = indices[title]

  sim_score = cosine_similarity(tf_idf_matrix[idx], tf_idf_matrix).flatten()
  similar_idx = sim_score.argsort()[::-1][1:n+1]
  return df['title'].iloc[similar_idx]


In [ ]:
reccomend('Spider Man')

In [ ]:
# import pickle
# pickle.dump(tf_idf,open("ifidf.pkl",'wb'))
# pickle.dump(tf_idf_matrix,open("ifidf_matrix.pkl",'wb'))
# pickle.dump(indices,open("indices.pkl",'wb'))
# df.to_pickle('df.pkl')